# Bronze — E-commerce Pedidos

## Objetivo

Realizar a ingestão incremental dos micro-lotes de `ecommerce_pedidos` disponibilizados no diretório RAW.

Para cada novo arquivo detectado pelo polling:

- ler o arquivo `.parquet` com Spark;
- preservar os dados recebidos, sem transformações de negócio;
- adicionar `bronze_ingested_at` com o timestamp da ingestão;
- adicionar `bronze_source_file` com o arquivo de origem;
- persistir os dados em formato Delta na camada Bronze;
- particionar por ano, mês, dia e hora da ingestão;
- registrar o arquivo processado no Manifest para evitar reprocessamento.

## Fluxo

`RAW → detecção de pendentes → preparação do micro-lote → Bronze → Manifest`


##  Autenticação e configurações

###  Bibliotecas utilizadas

As bibliotecas utilizadas no processo são carregadas nesta etapa.


In [0]:
# IMPORTS

import time
import pyarrow as pa

from pyspark.sql import functions as F
from pyspark.sql.types import (
    StructType,
    StructField,
    LongType,
    StringType,
    DoubleType
)

from deltalake import DeltaTable, write_deltalake

from azure.identity import ClientSecretCredential
from azure.storage.filedatalake import DataLakeServiceClient

### Configuração dos caminhos e parâmetros

As credenciais de acesso são obtidas por meio do notebook de autenticação.

Os dados da camada Bronze serão armazenados fisicamente no Data Lake, enquanto o histórico de ingestão será armazenado em uma estrutura de metadados separada.

In [0]:
%run ./00_autenticação

In [0]:
# CONFIGURAÇÃO OAUTH — Spark -> ADLS
# Prepara autenticação do Spark para leitura do Data Lake.
# NÃO grava dados.

storage_account = "internshipdatalake"

adls_options = {
    f"fs.azure.account.auth.type.{storage_account}.dfs.core.windows.net": "OAuth",

    f"fs.azure.account.oauth.provider.type.{storage_account}.dfs.core.windows.net":
        "org.apache.hadoop.fs.azurebfs.oauth2.ClientCredsTokenProvider",

    f"fs.azure.account.oauth2.client.id.{storage_account}.dfs.core.windows.net":
        client_id,

    f"fs.azure.account.oauth2.client.secret.{storage_account}.dfs.core.windows.net":
        client_secret,

    f"fs.azure.account.oauth2.client.endpoint.{storage_account}.dfs.core.windows.net":
        f"https://login.microsoftonline.com/{tenant_id}/oauth2/token"
}

print("Configuração OAuth preparada.")




In [0]:
# CAMINHOS OFICIAIS
# NÃO grava dados.

lake_container = "squad1"

bronze_path = (
    f"abfss://{lake_container}@"
    f"{storage_account}.dfs.core.windows.net/"
    "bronze/ecommerce_pedidos"
)

manifest_path = (
    f"abfss://{lake_container}@"
    f"{storage_account}.dfs.core.windows.net/"
    "metadata/ecommerce_pedidos_manifest"
)

print("Bronze:")
print(bronze_path)

print("\nManifest:")
print(manifest_path)

In [0]:
# CONEXÃO COM O RAW
# Prepara o acesso ao container de origem.
# NÃO grava dados.

credential = ClientSecretCredential(
    tenant_id=tenant_id,
    client_id=client_id,
    client_secret=client_secret
)

raw_service = DataLakeServiceClient(
    account_url="https://internshipdatalake.dfs.core.windows.net",
    credential=credential
)

raw_container = raw_service.get_file_system_client(
    file_system="raw"
)

In [0]:
# SCHEMA TÉCNICO — ecommerce_pedidos
# Os campos físicos TIMESTAMP(NANOS) do Parquet são lidos como LongType
# para preservar exatamente os valores recebidos no RAW.

schema_pedidos = StructType([
    StructField("id_pedido", LongType(), True),
    StructField("id_cliente", LongType(), True),
    StructField("id_endereco_entrega", LongType(), True),
    StructField("dt_pedido", LongType(), True),
    StructField("status_pedido", StringType(), True),
    StructField("valor_total", DoubleType(), True),
    StructField("valor_frete", DoubleType(), True),
    StructField("metodo_pagamento", StringType(), True),
    StructField("dt_previsao_entrega", LongType(), True),
    StructField("dt_ultima_atualizacao_status", LongType(), True)
])

In [0]:

# PREPARAÇÃO DO MICRO-LOTE
# READ-ONLY: lê o RAW e prepara o DataFrame Bronze.
# Não grava dados.

def preparar_micro_lote(arquivo_origem):

    raw_file_path = (
        f"abfss://raw@{storage_account}.dfs.core.windows.net/"
        f"{arquivo_origem}"
    )

    # 1. Leitura do Parquet diretamente com Spark
    df_raw = (
        spark.read
        .options(**adls_options)
        .schema(schema_pedidos)
        .parquet(raw_file_path)
    )

    # 2. Adição somente dos metadados técnicos da Bronze
    df_bronze = (
        df_raw
        .withColumn(
            "bronze_ingested_at",
            F.current_timestamp()
        )
        .withColumn(
            "bronze_source_file",
            F.lit(arquivo_origem)
        )
        .withColumn(
            "bronze_ingestion_year",
            F.year("bronze_ingested_at")
        )
        .withColumn(
            "bronze_ingestion_month",
            F.month("bronze_ingested_at")
        )
        .withColumn(
            "bronze_ingestion_day",
            F.dayofmonth("bronze_ingested_at")
        )
        .withColumn(
            "bronze_ingestion_hour",
            F.hour("bronze_ingested_at")
        )
    )

    return df_bronze

In [0]:
# PROCESSAMENTO DE UM MICRO-LOTE
# IMPORTANTE:
# Executar esta célula apenas DEFINE a função.
# Quando a função for CHAMADA, ela poderá gravar Bronze e Manifest.

def processar_micro_lote(arquivo_origem):

    # 1. Bloqueia escrita caso a Bronze existente
    # esteja com contrato incompatível
    validar_contrato_bronze()

    # 2. Confere idempotência pelo Manifest
    df_manifest = (
        spark.read
        .format("delta")
        .options(**adls_options)
        .load(manifest_path)
    )

    arquivo_ja_processado = (
        df_manifest
        .filter(F.col("arquivo_origem") == arquivo_origem)
        .limit(1)
        .count()
        > 0
    )

    if arquivo_ja_processado:
        print(f"Arquivo já processado: {arquivo_origem}")
        return

    # 3. RAW -> DataFrame Bronze
    df_bronze = preparar_micro_lote(arquivo_origem)

    qtd_registros = df_bronze.count()

    print(f"Arquivo: {arquivo_origem}")
    print(f"Registros encontrados: {qtd_registros}")

    # 4. Spark -> Arrow para persistência com delta-rs
    bronze_arrow = pa.Table.from_pandas(
        df_bronze.toPandas(),
        preserve_index=False
    )

    # 5. GRAVAÇÃO NA BRONZE
    write_deltalake(
        bronze_path,
        bronze_arrow,
        mode="append",
        partition_by=PARTICOES_BRONZE_ESPERADAS,
        storage_options=delta_storage_options
    )

    # 6. Registro técnico do Manifest
    manifest_registro = (
        spark.createDataFrame(
            [
                (
                    arquivo_origem,
                    qtd_registros,
                    qtd_registros
                )
            ],
            """
            arquivo_origem string,
            qtd_registros long,
            qtd_gravados_bronze long
            """
        )
        .withColumn(
            "processed_at",
            F.current_timestamp().cast("timestamp_ntz")
        )
        .select(
            "arquivo_origem",
            "processed_at",
            "qtd_registros",
            "qtd_gravados_bronze"
        )
    )

    manifest_arrow = pa.Table.from_pandas(
        manifest_registro.toPandas(),
        preserve_index=False
    )

    # 7. GRAVAÇÃO NO MANIFEST
    write_deltalake(
        manifest_path,
        manifest_arrow,
        mode="append",
        storage_options=delta_storage_options
    )

    print("✅ Micro-lote processado com sucesso.")

In [0]:
# BUSCA DE MICRO-LOTES PENDENTES
# READ-ONLY: compara o RAW com o Manifest.
# Não grava dados.

def buscar_micro_lotes_pendentes():

    # 1. Arquivos atualmente disponíveis no RAW
    arquivos_raw = {
        path.name
        for path in raw_container.get_paths(
            path="real-time-data",
            recursive=True
        )
        if (
            not path.is_directory
            and path.name.endswith("ecommerce_pedidos.parquet")
        )
    }

    # 2. Arquivos já registrados no Manifest
    df_manifest = (
        spark.read
        .format("delta")
        .options(**adls_options)
        .load(manifest_path)
    )

    arquivos_processados = {
        row["arquivo_origem"]
        for row in (
            df_manifest
            .select("arquivo_origem")
            .distinct()
            .collect()
        )
    }

    # 3. RAW - Manifest = pendentes
    arquivos_pendentes = sorted(
        arquivos_raw - arquivos_processados
    )

    return arquivos_pendentes

In [0]:
# CONFIGURAÇÃO DELTA-RS — escrita direta no ADLS
# Apenas prepara autenticação.
# NÃO grava dados.

delta_storage_options = {
    "azure_storage_client_id": client_id,
    "azure_storage_client_secret": client_secret,
    "azure_storage_tenant_id": tenant_id,
}

In [0]:
# VALIDAÇÃO DO CONTRATO DA BRONZE
# READ-ONLY: verifica o particionamento atual.
# NÃO grava nem altera dados.

PARTICOES_BRONZE_ESPERADAS = [
    "bronze_ingestion_year",
    "bronze_ingestion_month",
    "bronze_ingestion_day",
    "bronze_ingestion_hour"
]


def validar_contrato_bronze():

    tabela_bronze = DeltaTable(
        bronze_path,
        storage_options=delta_storage_options
    )

    particoes_atuais = (
        tabela_bronze
        .metadata()
        .partition_columns
    )

    print("Partições atuais:", particoes_atuais)
    print("Partições esperadas:", PARTICOES_BRONZE_ESPERADAS)

    if particoes_atuais != PARTICOES_BRONZE_ESPERADAS:
        raise RuntimeError(
            "Bronze existente incompatível com o contrato atual. "
            "Nenhuma escrita será realizada."
        )

    print("✅ Contrato da Bronze compatível.")

In [0]:
# POLLING CONTROLADO
# IMPORTANTE:
# Executar esta célula apenas DEFINE a função.
# A escrita só acontece quando executar_polling(...) for chamada.

def executar_polling(
    ciclos=3,
    intervalo_segundos=30
):

    for ciclo in range(1, ciclos + 1):

        print(f"\n===== CICLO {ciclo}/{ciclos} =====")

        arquivos_pendentes = buscar_micro_lotes_pendentes()

        print(
            f"Arquivos pendentes encontrados: "
            f"{len(arquivos_pendentes)}"
        )

        for arquivo in arquivos_pendentes:
            print(f"Processando: {arquivo}")
            processar_micro_lote(arquivo)

        if ciclo < ciclos:
            print(
                f"Aguardando "
                f"{intervalo_segundos} segundos..."
            )

            time.sleep(intervalo_segundos)

    print("\nPolling finalizado.")

In [0]:
# EXECUÇÃO DO PIPELINE — JANELA DE POLLING
# WRITE: novos arquivos encontrados poderão ser gravados na Bronze e no Manifest.

executar_polling(
    ciclos=10,
    intervalo_segundos=30
)

In [0]:
# VALIDAÇÃO FINAL DA BRONZE — READ-ONLY
# NÃO grava, altera ou apaga dados.

df_bronze_final = (
    spark.read
    .format("delta")
    .options(**adls_options)
    .load(bronze_path)
)

df_manifest_final = (
    spark.read
    .format("delta")
    .options(**adls_options)
    .load(manifest_path)
)

tabela_bronze_final = DeltaTable(
    bronze_path,
    storage_options=delta_storage_options
)

particoes_finais = (
    tabela_bronze_final
    .metadata()
    .partition_columns
)

qtd_bronze = df_bronze_final.count()

qtd_arquivos_bronze = (
    df_bronze_final
    .select("bronze_source_file")
    .distinct()
    .count()
)

qtd_auditoria_nula = (
    df_bronze_final
    .filter(
        F.col("bronze_ingested_at").isNull()
        | F.col("bronze_source_file").isNull()
    )
    .count()
)

qtd_particoes_nulas = (
    df_bronze_final
    .filter(
        F.col("bronze_ingestion_year").isNull()
        | F.col("bronze_ingestion_month").isNull()
        | F.col("bronze_ingestion_day").isNull()
        | F.col("bronze_ingestion_hour").isNull()
    )
    .count()
)

qtd_manifest = df_manifest_final.count()

qtd_arquivos_manifest = (
    df_manifest_final
    .select("arquivo_origem")
    .distinct()
    .count()
)

print("=== BRONZE FINAL ===")
print("Registros:", qtd_bronze)
print("Arquivos de origem distintos:", qtd_arquivos_bronze)
print("Auditoria nula:", qtd_auditoria_nula)
print("Partições nulas:", qtd_particoes_nulas)
print("Particionamento:", particoes_finais)

print("\n=== MANIFEST FINAL ===")
print("Registros:", qtd_manifest)
print("Arquivos distintos:", qtd_arquivos_manifest)

print("\n=== SCHEMA BRONZE ===")
df_bronze_final.printSchema()

# Validações automáticas
assert qtd_auditoria_nula == 0, "Há metadados de auditoria nulos."
assert qtd_particoes_nulas == 0, "Há partições de ingestão nulas."
assert particoes_finais == PARTICOES_BRONZE_ESPERADAS, "Particionamento incorreto."
assert qtd_manifest == qtd_arquivos_manifest, "Há arquivos duplicados no Manifest."

print("\n✅ VALIDAÇÃO FINAL APROVADA.")